# Analytics Village — CH01: Open for Business
### Village Fresh Supermarket · Star Schema (Data Warehouse)

---

You're a data analyst hired by **Khun Somchai**, owner of Village Fresh Supermarket.

This notebook uses the **star schema database** (`village_star.db`) — the data is pre-joined into fact and dimension tables so you can focus on analysis, not data preparation.

**Dimensions:** `dim_customer`, `dim_product`, `dim_store`, `dim_date`

**Facts:** `fact_sales`, `fact_daily_store`, `fact_inventory`, `fact_customer_lifecycle`, `fact_competitor_visits`

## Setup

In [ ]:
!pip install pandas tabulate -q
import os
if not os.path.exists('analytics-village'):
    !git clone https://github.com/thanachart/analytics-village.git
else:
    !cd analytics-village && git pull -q

import sys
sys.path.insert(0, 'analytics-village')

from student import Challenge, Decision
ch = Challenge.load('ch01', data_dir='analytics-village/challenges/ch01/data', db_name='village_star.db')

## Brief & Questions

In [ ]:
ch.brief()

In [ ]:
# View challenge questions
ch.questions()

## Explore the Star Schema

The star schema has **fact tables** (measures/events) and **dimension tables** (context). Each fact row already contains the keys to join with dimensions.

| Fact Table | What it measures |
|---|---|
| `fact_sales` | One row per item sold — revenue, qty, satisfaction, stockouts |
| `fact_daily_store` | Daily store KPIs — revenue, customers, stockouts, waste |
| `fact_inventory` | Daily stock snapshot per product |
| `fact_customer_lifecycle` | State transitions (retained → at-risk → churned) |
| `fact_competitor_visits` | Purchases at competitor stores |

In [ ]:
ch.db.tables()

In [ ]:
# fact_sales — the main analysis table
ch.db.query('SELECT * FROM fact_sales LIMIT 5')

In [ ]:
# Revenue by day (already flat — no joins needed!)
ch.db.query('''
    SELECT date_key, day_of_week, is_payday_week,
           COUNT(*) AS items_sold,
           SUM(line_total_thb) AS revenue,
           COUNT(DISTINCT customer_key) AS unique_customers,
           AVG(satisfaction_score) AS avg_satisfaction
    FROM fact_sales
    GROUP BY date_key
    ORDER BY date_key
    LIMIT 15
''')

In [ ]:
# Revenue by product category
ch.db.query('''
    SELECT p.category, COUNT(*) AS items_sold,
           SUM(fs.line_total_thb) AS revenue,
           SUM(fs.had_stockout) AS stockouts
    FROM fact_sales fs
    JOIN dim_product p ON fs.product_key = p.product_key
    GROUP BY p.category
    ORDER BY revenue DESC
''')

In [ ]:
# Revenue by customer income segment
ch.db.query('''
    SELECT c.income_bracket,
           COUNT(DISTINCT fs.customer_key) AS customers,
           SUM(fs.line_total_thb) AS total_revenue,
           AVG(fs.transaction_total_thb) AS avg_basket,
           AVG(fs.satisfaction_score) AS avg_satisfaction
    FROM fact_sales fs
    JOIN dim_customer c ON fs.customer_key = c.customer_key
    GROUP BY c.income_bracket
    ORDER BY total_revenue DESC
''')

In [ ]:
# Day-of-week pattern
ch.db.query('''
    SELECT day_of_week,
           COUNT(DISTINCT transaction_id) AS transactions,
           SUM(line_total_thb) AS revenue,
           COUNT(DISTINCT customer_key) AS customers
    FROM fact_sales
    GROUP BY day_of_week
''')

In [ ]:
# Customer lifecycle events
ch.db.query('''
    SELECT new_state, trigger_reason, COUNT(*) AS events
    FROM fact_customer_lifecycle
    GROUP BY new_state, trigger_reason
    ORDER BY events DESC
''')

In [ ]:
# Daily store performance
ch.db.query('''
    SELECT date_key, revenue_thb, transaction_count, unique_customers,
           avg_basket_thb, stockout_count, waste_cost_thb
    FROM fact_daily_store
    ORDER BY date_key
    LIMIT 15
''')

## Your Analysis

With the star schema, you can jump straight into analysis:
- Revenue trends and seasonality
- Customer segmentation by spending patterns
- Product performance and stockout impact
- Churn analysis and risk factors
- Payday vs non-payday spending

In [ ]:
# YOUR ANALYSIS HERE


## Build & Submit

In [ ]:
d = Decision('ch01', student_ids=['YOUR_STUDENT_ID'])

d.set_finding(
    headline='YOUR KEY FINDING (30-200 chars)',
    evidence='How you found it and what data supports it (50+ chars)',
)
d.set_methodology(approach='Your analytical approach')
d.set_recommendation(
    action_type='winback',
    recommendation='Your specific recommendation',
    target_description='Who or what is targeted',
    target_size=0,
    budget_thb=0,
    expected_outcome='What you expect (quantified)',
    timeline_days=14,
    success_metric='How to measure success',
    risk='What could go wrong (20+ chars)',
)

d.validate()

In [ ]:
filepath = d.export(output_dir='analytics-village/submissions/ch01')
print(f'Saved: {filepath}')
print('Submit: upload to https://github.com/thanachart/analytics-village/tree/main/submissions/ch01')